# 06 · Checking each label against the text

CTI2TTP reported precision rising from 55.55% to 82.15% when a verification step checked each predicted label. Two versions
are compared on the funnel's labels:

- **quote filter:** keep a label only if the quote the model gave really appears in the text. No extra model call.
- **second reading:** another call asks whether each label is justified, and needs a real quote to accept it.

**Limit.** Neither test set marks which passage supports a label, so this measures label accuracy, not whether the right
passage was found.

**What would count as what, written before running.**
- Expected: precision rises and recall falls. It helps overall only if F1 rises, with a 95% interval excluding zero.
- On HumSet, missing labels are not always wrong, so the within-project scores are the fairer view of precision.


In [1]:
import pandas as pd

from tiered_taxonomy import experiment as ex
from tiered_taxonomy import runs
from tiered_taxonomy.config import data_root, load_config

root = data_root()
config = load_config()
seed = config["seed"]
samples = config["samples"]


def sample_for(data):
    if data.name == ex.EURLEX:
        settings = samples["eurlex"]
        return ex.sample_eurlex(
            data,
            settings["normal_documents"],
            settings["long_documents"],
            settings["long_threshold_words"],
            seed,
        )
    return ex.sample_humset(data, samples["humset"]["per_language"], seed)


def table(results):
    rows = []
    for name, result in results.items():
        scores = result["scores"]
        row = {
            "method": name,
            "labels per item": scores["labels_per_item"],
            "exact F1": round(scores["exact"]["f1"], 3),
            "exact precision": round(scores["exact"]["precision"], 3),
            "exact recall": round(scores["exact"]["recall"], 3),
        }
        row.update({f"tier {tier} F1": round(value["f1"], 3) for tier, value in scores["tiers"].items()})
        row.update({f"{band} recall": value["recall"] for band, value in scores["rarity"].items()})
        if "run" in result:
            run = result["run"]
            row.update(
                {
                    "calls per item": round(run["calls"] / run["items"], 1),
                    "cost per 1000 items ($)": round(1000 * run["cost_usd"] / run["items"], 2),
                    "malformed": run["malformed"],
                    "invalid": run["invalid"],
                }
            )
        rows.append(row)
    return pd.DataFrame(rows)

/Users/matthewharris/git/nvidia-autoontology-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
eurlex = ex.load_eurlex(root)
humset = ex.load_humset(root)
benches = {data.name: runs.make_bench(data, root, config) for data in (eurlex, humset)}
sets = {data.name: data for data in (eurlex, humset)}
items = {name: sample_for(data) for name, data in sets.items()}
for name, bench in benches.items():
    runs.load_cards(bench)
    runs.embed_inputs(bench, items[name])
used = ex.project_labels([*humset.train, *humset.test], humset.taxonomy)
results = {}

In [3]:
from tiered_taxonomy.metrics import micro_f1, paired_difference

for name in sets:
    results[name] = runs.evidence_runs(benches[name], "funnel_cards", items[name])

## MultiEURLEX

In [4]:
table(results[ex.EURLEX])

,method,labels per item,exact F1,exact precision,exact recall,tier 1 F1,tier 2 F1,tier 3 F1,common recall,rare recall,unseen recall,calls per item,cost per 1000 items ($),malformed,invalid
0,funnel_cards,3.76,0.165,0.226,0.130,0.414,0.307,0.260,0.1248,0.1774,0.2250,1.0,0.60,4,0
1,funnel_cards_quote_filter,2.73,0.148,0.250,0.105,0.371,0.275,0.234,0.1004,0.1398,0.1833,1.0,0.60,4,0
2,funnel_cards_checked,2.98,0.147,0.234,0.107,0.363,0.263,0.221,0.1018,0.1505,0.2000,2.0,0.79,4,0


## HumSet

In [5]:
table(results[ex.HUMSET])

,method,labels per item,exact F1,exact precision,exact recall,tier 1 F1,tier 2 F1,common recall,rare recall,calls per item,cost per 1000 items ($),malformed,invalid
0,funnel_cards,1.75,0.395,0.463,0.344,0.445,0.280,0.3444,0.3226,1.0,0.3,0,0
1,funnel_cards_quote_filter,1.47,0.352,0.458,0.286,0.402,0.242,0.2853,0.3065,1.0,0.3,0,0
2,funnel_cards_checked,1.57,0.371,0.463,0.309,0.418,0.262,0.3095,0.2903,2.0,0.4,0,0


In [6]:
differences = {}
for name in sets:
    sample = items[name]
    ids = [item.id for item in sample]
    gold = ex.gold_exact(sample)

    def get(method, name=name, ids=ids):
        return ex.load_predictions(root, name, method, ids)

    base = get("funnel_cards")
    differences[name] = {
        "quote filter minus plain": paired_difference(micro_f1, gold, get("funnel_cards_quote_filter"), base),
        "second reading minus plain": paired_difference(micro_f1, gold, get("funnel_cards_checked"), base),
    }
within = {
    method: ex.evaluate_within_project(
        humset,
        items[ex.HUMSET],
        ex.load_predictions(root, ex.HUMSET, method, [i.id for i in items[ex.HUMSET]]),
        used,
    )
    for method in results[ex.HUMSET]
}
ex.save_results(
    "06_evidence",
    {
        "results": results,
        "differences": differences,
        "humset_within_project": within,
        "models": config["models"],
    },
)
differences

{'eurlex': {'quote filter minus plain': (-0.01764891524317866,
   -0.02441167712681462,
   -0.011450842817181556),
  'second reading minus plain': (-0.018431533171906572,
   -0.024669817121592396,
   -0.012885567432364452)},
 'humset': {'quote filter minus plain': (-0.04041933793946145,
   -0.04781009036084627,
   -0.032860208160929764),
  'second reading minus plain': (-0.0224142961971035,
   -0.02891733793334881,
   -0.01677028270611444)}}